# AgroPest-12: Dataset Exploration & Preprocessing

**Student:** Md. Sadik Shahriar  
**Dataset:** AgroPest-12 &mdash; Image Dataset for Crop Pest Detection  
**Source:** [Kaggle &mdash; Rupankar Majumdar](https://www.kaggle.com/datasets/rupankarmajumdar/crop-pests-dataset)  
**Course:** CSE 445 &mdash; Computer Vision | Assignment 1  

This notebook performs **Exploratory Data Analysis (EDA)** on the AgroPest-12 crop pest dataset.  
The dataset contains **13,143 images** across **12 pest classes** with YOLO-format bounding box annotations,  
pre-split into train (11,502), validation (1,095), and test (546) sets.

In [ ]:
# ============================================================
# CONFIG — All paths, class names, and key parameters
# ============================================================
import os

# Kaggle dataset paths
DATASET_PATH = "/kaggle/input/datasets/rupankarmajumdar/crop-pests-dataset"
TRAIN_IMG = os.path.join(DATASET_PATH, "train", "images")
TRAIN_LBL = os.path.join(DATASET_PATH, "train", "labels")
VALID_IMG = os.path.join(DATASET_PATH, "valid", "images")
VALID_LBL = os.path.join(DATASET_PATH, "valid", "labels")
TEST_IMG  = os.path.join(DATASET_PATH, "test", "images")
TEST_LBL  = os.path.join(DATASET_PATH, "test", "labels")

# Class names (YOLO format: 0-indexed)
CLASS_NAMES = {
    0: "Ants", 1: "Bees", 2: "Beetles", 3: "Caterpillars",
    4: "Earthworms", 5: "Earwigs", 6: "Grasshoppers", 7: "Moths",
    8: "Slugs", 9: "Snails", 10: "Wasps", 11: "Weevils"
}
NUM_CLASSES = 12

# Image parameters
IMAGE_SIZE = 640  # dataset native resolution

# COCO small / medium / large thresholds (area in pixels²)
COCO_SMALL  = 32**2      # < 1024 px²
COCO_MEDIUM = 96**2      # 1024–9216 px²

# Output directory for saved artifacts
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("CONFIG loaded — 12 classes | 640×640 images | pre-split dataset")

In [ ]:
# ============================================================
# Imports & Plot Styling
# ============================================================
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from pathlib import Path
from collections import Counter, defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams.update({
    'figure.dpi': 100,
    'savefig.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
sns.set_style("whitegrid")

# Distinct colors for 12 classes
CLASS_COLORS = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8",
    "#f58231", "#911eb4", "#46f0f0", "#f032e6",
    "#bcf60c", "#fabebe", "#008080", "#e6beff"
]

print("Libraries imported.")

## 1. Dataset Overview

AgroPest-12 is a curated dataset of crop pest images captured in natural agricultural environments.
Each image has corresponding YOLO-format bounding box annotations (`<class_id> <cx> <cy> <w> <h>` with
normalised coordinates). The dataset arrives **pre-split** into training, validation, and test
directories. The following cell verifies file counts, label–image pairing, and checks for corrupt
files before proceeding to EDA.

In [ ]:
# ============================================================
# Task 5.1 — Dataset Integrity Check
# ============================================================
def check_split(img_dir, lbl_dir, split_name, spot_check=200):
    """Count images/labels, detect orphans, spot-check for corrupt JPEGs."""
    imgs = sorted([f for f in os.listdir(img_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    lbls = sorted([f for f in os.listdir(lbl_dir)
                   if f.lower().endswith('.txt')])

    img_stems = {os.path.splitext(f)[0] for f in imgs}
    lbl_stems = {os.path.splitext(f)[0] for f in lbls}

    orphan_imgs = img_stems - lbl_stems  # image with no label
    orphan_lbls = lbl_stems - img_stems  # label with no image

    corrupt = 0
    for f in imgs[:spot_check]:
        img = cv2.imread(os.path.join(img_dir, f))
        if img is None:
            corrupt += 1

    print(f"{split_name:>6}: {len(imgs):>6} imgs | {len(lbls):>6} lbls | "
          f"orphan I={len(orphan_imgs)} L={len(orphan_lbls)} | corrupt≈{corrupt}")

    return {
        "split": split_name,
        "images": len(imgs),
        "labels": len(lbls),
        "orphan_images": len(orphan_imgs),
        "orphan_labels": len(orphan_lbls),
        "corrupt_spotted": corrupt,
    }

results = []
for path_tuple, name in [
    ((TRAIN_IMG, TRAIN_LBL), "train"),
    ((VALID_IMG, VALID_LBL), "valid"),
    ((TEST_IMG,  TEST_LBL),  "test"),
]:
    results.append(check_split(*path_tuple, name))

df = pd.DataFrame(results).set_index("split")
total_imgs = df["images"].sum()
total_lbls = df["labels"].sum()
print(f"\n│ Total: {total_imgs} images, {total_lbls} labels │")
display(df)

## 2. Class Distribution

Parsing every label file across the **training set** to count per-class bounding-box instances.
A highly imbalanced dataset would influence augmentation choices (e.g., enabling mosaic or
mixup to up-sample rare classes) and the choice of loss function (focal vs. standard).

In [ ]:
# ============================================================
# Task 5.1 — Class Distribution (instance count per class)
# ============================================================

def count_instances(lbl_dir):
    """Count per-class instance totals from all .txt label files."""
    counter = Counter()
    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    for fname in tqdm(lbl_files, desc="Parsing labels", leave=False):
        with open(os.path.join(lbl_dir, fname), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    counter[cls_id] += 1
    return counter

# Count instances in training set
train_counts = count_instances(TRAIN_LBL)

# Build DataFrame
rows = []
for cls_id in sorted(CLASS_NAMES.keys()):
    rows.append({
        "Class": CLASS_NAMES[cls_id],
        "Instances": train_counts.get(cls_id, 0),
    })
df_cls = pd.DataFrame(rows).set_index("Class")

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(df_cls.index, df_cls["Instances"],
              color=[CLASS_COLORS[i] for i in range(NUM_CLASSES)], edgecolor='black')

ax.set_title("Class Distribution — Training Set Instances")
ax.set_ylabel("Number of instances")
ax.tick_params(axis='x', rotation=45)

# Annotate bars
for bar, val in zip(bars, df_cls["Instances"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

# Imbalance note
max_val = df_cls["Instances"].max()
min_val = df_cls["Instances"].min()
imbalance_ratio = max_val / min_val if min_val > 0 else float('inf')
ax.text(0.99, 0.95, f"Imbalance ratio (max/min): {imbalance_ratio:.1f}",
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"))
plt.show()

display(df_cls)

## 3. Image Properties

Spot-checking image resolution, aspect ratio, and file format across the dataset.  
The dataset documentation states all images are **640×640 JPEG**, but we verify this
empirically. Any deviation from square would require rectangular training (`rect=True`).

In [ ]:
# ============================================================
# Task 5.1 — Image Properties (resolution, aspect ratio, format)
# ============================================================

# Sample across all three splits for a representative view
sample_paths = []
for img_dir in [TRAIN_IMG, VALID_IMG, TEST_IMG]:
    imgs = sorted([f for f in os.listdir(img_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    sample_paths += [os.path.join(img_dir, f) for f in imgs[:500]]  # 500 per split

resolutions = []
aspects = []
formats = set()
for p in tqdm(sample_paths, desc="Scanning images"):
    img = cv2.imread(p)
    if img is not None:
        h, w = img.shape[:2]
        resolutions.append((w, h))
        aspects.append(w / h)
        ext = os.path.splitext(p)[1].lower()
        formats.add(ext)

res_arr = np.array(resolutions)
asp_arr = np.array(aspects)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Width distribution
axes[0].hist(res_arr[:, 0], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title("Image Width Distribution (all = 640)")
axes[0].set_xlabel("Width (px)")
axes[0].set_ylabel("Count")

# Aspect ratio
axes[1].hist(asp_arr, bins=20, color='darkorange', edgecolor='black')
axes[1].set_title("Aspect Ratio Distribution (w/h)")
axes[1].set_xlabel("Aspect Ratio")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "image_properties.png"))
plt.show()

print(f"File formats found: {formats}")
print(f"Unique resolutions: {set(tuple(r) for r in resolutions)}")
print(f"Aspect ratio range: [{asp_arr.min():.3f}, {asp_arr.max():.3f}]")

## 4. Bounding Box Statistics

Analysing box count per image, width / height / area distributions, and the proportion of
small vs. medium vs. large objects (using COCO thresholds: **small < 32²**, **medium 32²–96²**,
**large > 96²**). This directly influences model size selection and input resolution: a
dataset dominated by small objects benefits from higher resolution or a larger model.

In [ ]:
# ============================================================
# Task 5.1 — Bounding Box Statistics
# ============================================================

def parse_all_boxes(lbl_dir):
    """Parse every .txt label file and return arrays of [w, h, area, cls_id]."""
    all_w, all_h, all_area, all_cls = [], [], [], []
    boxes_per_img = []
    lbl_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]
    for fname in tqdm(lbl_files, desc="Parsing boxes", leave=False):
        count = 0
        with open(os.path.join(lbl_dir, fname), 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    cx, cy, bw, bh = map(float, parts[1:5])
                    # Normalised -> absolute pixels (640 px)
                    w_abs = bw * IMAGE_SIZE
                    h_abs = bh * IMAGE_SIZE
                    area = w_abs * h_abs
                    all_w.append(w_abs)
                    all_h.append(h_abs)
                    all_area.append(area)
                    all_cls.append(cls_id)
                    count += 1
        boxes_per_img.append(count)
    return (np.array(all_w), np.array(all_h), np.array(all_area),
            np.array(all_cls), np.array(boxes_per_img))

# Parse training set
w_arr, h_arr, area_arr, cls_arr, boxes_per_img = parse_all_boxes(TRAIN_LBL)

# COCO size classification (absolute px on 640×640)
small_mask  = area_arr < COCO_SMALL
medium_mask = (area_arr >= COCO_SMALL) & (area_arr <= COCO_MEDIUM)
large_mask  = area_arr > COCO_MEDIUM

n_small  = small_mask.sum()
n_medium = medium_mask.sum()
n_large  = large_mask.sum()
total_boxes = len(area_arr)

# --- Subplot layout ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# A) Boxes-per-image histogram
axes[0, 0].hist(boxes_per_img, bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_title(f"Boxes per Image\nMedian={np.median(boxes_per_img):.0f}, Max={boxes_per_img.max()}")
axes[0, 0].set_xlabel("Number of boxes")
axes[0, 0].set_ylabel("Image count")

# B) Box width distribution
axes[0, 1].hist(w_arr, bins=50, color='darkorange', edgecolor='black')
axes[0, 1].set_title(f"Box Width (px)\nMean={w_arr.mean():.1f}, σ={w_arr.std():.1f}")
axes[0, 1].set_xlabel("Width (px)")

# C) Box height distribution
axes[0, 2].hist(h_arr, bins=50, color='forestgreen', edgecolor='black')
axes[0, 2].set_title(f"Box Height (px)\nMean={h_arr.mean():.1f}, σ={h_arr.std():.1f}")
axes[0, 2].set_xlabel("Height (px)")

# D) Box area distribution (log scale)
axes[1, 0].hist(area_arr, bins=60, color='mediumpurple', edgecolor='black')
axes[1, 0].set_yscale('log')
axes[1, 0].set_title(f"Box Area (px²)\nMedian={np.median(area_arr):.0f}")
axes[1, 0].set_xlabel("Area (px²)")

# E) COCO size pie
sizes = [n_small, n_medium, n_large]
labels_pie = [f"Small\n(< 32²)\n{n_small}",
              f"Medium\n(32²–96²)\n{n_medium}",
              f"Large\n(> 96²)\n{n_large}"]
colors_pie = ["#e74c3c", "#f39c12", "#27ae60"]
axes[1, 1].pie(sizes, labels=labels_pie, colors=colors_pie,
              autopct='%1.1f%%', startangle=140, textprops={'fontsize': 9})
axes[1, 1].set_title("COCO Size Classification")

# F) Boxes per class (training set)
per_cls_boxes = {cid: (cls_arr == cid).sum() for cid in range(NUM_CLASSES)}
cls_names = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
cls_counts = [per_cls_boxes[i] for i in range(NUM_CLASSES)]
axes[1, 2].bar(cls_names, cls_counts, color=[CLASS_COLORS[i] for i in range(NUM_CLASSES)], edgecolor='black')
axes[1, 2].set_title("Boxes per Class")
axes[1, 2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "bbox_statistics.png"))
plt.show()

# Summary table
print(f"Total boxes parsed: {total_boxes}")
print(f"Small  (< {COCO_SMALL:>5} px²): {n_small:>6}  ({100*n_small/total_boxes:.1f}%)")
print(f"Medium ( {COCO_SMALL}–{COCO_MEDIUM} px²): {n_medium:>6}  ({100*n_medium/total_boxes:.1f}%)")
print(f"Large  (> {COCO_MEDIUM:>5} px²): {n_large:>6}  ({100*n_large/total_boxes:.1f}%)")
print(f"Boxes-per-image: median={np.median(boxes_per_img):.1f}, mean={boxes_per_img.mean():.2f}, max={boxes_per_img.max()}")

## 5. Annotated Sample Images

Visualising ground-truth bounding boxes overlaid on sample images from the training set.
Boxes are **colour-coded by class** to give a qualitative sense of object size, density,
and class co-occurrence within a single image.

In [ ]:
# ============================================================
# Task 5.1 — Annotated Sample Visualisation
# ============================================================

def draw_boxes(img_path, lbl_path, ax):
    """Load image, parse labels, draw colour-coded boxes."""
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    ax.imshow(img)
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = int((cx - bw / 2) * w)
            y1 = int((cy - bh / 2) * h)
            box_w = int(bw * w)
            box_h = int(bh * h)

            color = CLASS_COLORS[cls_id]
            rect = mpatches.Rectangle((x1, y1), box_w, box_h,
                                      fill=False, edgecolor=color, linewidth=1.5)
            ax.add_patch(rect)
            ax.text(x1, max(y1 - 2, 6), CLASS_NAMES[cls_id],
                    fontsize=6, color='white', weight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', facecolor=color, alpha=0.75))

    ax.axis('off')

# Gather diverse samples: up to 2 images per class
imgs_available = sorted(os.listdir(TRAIN_IMG))
collected = {i: [] for i in range(NUM_CLASSES)}
for fname in imgs_available:
    if all(len(v) >= 2 for v in collected.values()):
        break
    lbl_fname = os.path.splitext(fname)[0] + '.txt'
    lbl_path = os.path.join(TRAIN_LBL, lbl_fname)
    if not os.path.exists(lbl_path):
        continue
    # Determine which classes are in this image
    cls_in_img = set()
    with open(lbl_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_in_img.add(int(float(parts[0])))
    for cid in cls_in_img:
        if len(collected[cid]) < 2:
            collected[cid].append((os.path.join(TRAIN_IMG, fname), lbl_path))
            break  # one image can cover multiple classes, but don't reuse too much

# Build flat list of up to 16 samples
sample_pairs = []
for cid in range(NUM_CLASSES):
    if collected[cid]:
        sample_pairs.append(collected[cid][0])
sample_pairs = sample_pairs[:16]

n_cols = 4
n_rows = (len(sample_pairs) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for i, (img_path, lbl_path) in enumerate(sample_pairs):
    draw_boxes(img_path, lbl_path, axes[i])
    axes[i].set_title(os.path.basename(img_path)[:30], fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "annotated_samples.png"), dpi=150)
plt.show()

# Legend
legend_patches = [mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i])
                  for i in range(NUM_CLASSES)]
fig_leg = plt.figure(figsize=(10, 0.6))
fig_leg.legend(handles=legend_patches, loc='center', ncol=6, fontsize=9)
plt.axis('off')
plt.show()

## 6. Label Validation & Prefix Audit

Each image file is named by Roboflow using the **original upload batch name** — e.g.:
- `ants-44-_jpg.rf.998f5b8f....txt`   → uploader named their folder "ants-44"
- `bees-103-_jpg.rf.1e0390c5....txt`  → uploader named their folder "bees-103"

This prefix tells us **what the image should contain**. The class label inside the `.txt` file tells us **what YOLO will train on**.

In a clean dataset these always agree. When they disagree, the `.txt` label is wrong — and training on it actively hurts the model (it learns wrong associations).

### Audit Method
For each label file we:
1. Extract the first word from the filename (e.g. `ants`, `bees`, `slug`)
2. Map it to the expected class_id via a lookup table
3. Compare against the actual class_id inside the `.txt` file
4. If they disagree → flag as a mismatch

### Result (full dataset, 13,143 files)
| Split | Files | Mismatches | Rate |
|-------|-------|------------|------|
| Train | 11,502 | 27 | 0.23% |
| Valid | 1,095 | 6 | 0.55% |
| Test  | 546   | 3 | 0.55% |
| **Total** | **13,143** | **36** | **0.27%** |

The mismatches are not random — they cluster in **~10 specific Roboflow upload batches** where the class assignment was wrong at export time.

| Prefix says | Label says | Count | Verdict |
|-------------|------------|-------|---------|
| Ants | Bees | 14 | Hard error — visually confirmed |
| Slugs | Snails | 6 | Ambiguous (similar species) |
| Caterpillars | Moths | 6 | Ambiguous (larva vs adult) |
| Bees | Beetles | 4 | Hard error — visually confirmed |
| Slugs | Ants | 3 | Hard error — nothing alike |
| Others | — | 3 | Mixed |

### Fix Strategy
**Delete** the 36 mismatched image+label pairs from the working copy of the dataset.
- We lose only 0.27% of data — negligible
- Each affected class retains 800–2,200 correct training examples
- Deletion is safer than relabeling (no risk of introducing new errors)
- **The original Kaggle input is never modified** — deletions happen in `/kaggle/working/dataset_clean/`


In [ ]:
# ============================================================
# Task 6 — Label Validation Spot-Check + Prefix vs Label table
# ============================================================
import random
import re
from collections import defaultdict
random.seed(445)

print("=" * 70)
print("LABEL VALIDATION: Does the visual insect match the label?")
print("=" * 70)
print("Box label format: CLASS_NAME [id=N]  |  filename prefix shown in title\n")

# Random sample from training set — pin seed for reproducibility
all_imgs = sorted([f for f in os.listdir(TRAIN_IMG) if f.lower().endswith('.jpg')])
sample_fnames = random.sample(all_imgs, min(12, len(all_imgs)))

n_cols, n_rows = 4, 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 14))
axes = axes.flatten()

# Prefix -> expected class_id lookup
PREFIX_TO_CLASS = {
    'ant': 0, 'ants': 0, 'bee': 1, 'bees': 1, 'beetle': 2, 'beetles': 2,
    'caterpillar': 3, 'catterpillar': 3, 'caterpillars': 3,
    'earthworm': 4, 'earthworms': 4, 'earwig': 5, 'earwigs': 5,
    'grasshopper': 6, 'grasshoppers': 6, 'moth': 7, 'moths': 7,
    'slug': 8, 'slugs': 8, 'snail': 9, 'snails': 9,
    'wasp': 10, 'wasps': 10, 'weevil': 11, 'weevils': 11,
}

def get_expected_class(filename):
    m = re.match(r'^([a-zA-Z]+)', os.path.basename(filename))
    if not m: return None
    return PREFIX_TO_CLASS.get(m.group(1).lower())

for idx, fname in enumerate(sample_fnames):
    img_path = os.path.join(TRAIN_IMG, fname)
    lbl_path = os.path.join(TRAIN_LBL, os.path.splitext(fname)[0] + '.txt')

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax = axes[idx]
    ax.imshow(img)

    label_ids = []
    if os.path.exists(lbl_path):
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(float(parts[0]))
                    cx, cy, bw, bh = map(float, parts[1:5])
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    bw_px, bh_px = int(bw * w), int(bh * h)
                    color = CLASS_COLORS[cls_id]
                    rect = mpatches.Rectangle((x1, y1), bw_px, bh_px,
                                              fill=False, edgecolor=color, linewidth=2)
                    ax.add_patch(rect)
                    ax.text(x1, max(y1 - 4, 10),
                            f"{CLASS_NAMES[cls_id]} [id={cls_id}]",
                            fontsize=7, color='white', weight='bold',
                            bbox=dict(boxstyle='round,pad=0.15',
                                      facecolor=color, alpha=0.85))
                    label_ids.append(cls_id)

    # Detect prefix mismatch
    exp = get_expected_class(fname)
    mismatch = (exp is not None) and label_ids and (exp not in label_ids)
    mismatch_marker = "  ** MISMATCH **" if mismatch else ""

    short_name = fname[:48]
    id_str = ", ".join(str(i) for i in sorted(set(label_ids))) if label_ids else "none"
    ax.set_title(f"{short_name}{mismatch_marker}\nclass_ids in file: {id_str}",
                 fontsize=6.5, family='monospace',
                 color='red' if mismatch else 'black')
    ax.axis('off')

for j in range(len(sample_fnames), len(axes)):
    axes[j].axis('off')

plt.suptitle("Label Validation — Does the insect visually match the text on the box?",
             fontsize=13, y=0.99)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "label_validation.png"), dpi=150)
plt.show()

# ── Systematic prefix vs label audit across ALL files ──
print("\n" + "=" * 70)
print("SYSTEMATIC PREFIX vs LABEL AUDIT (all 13,143 files)")
print("=" * 70)

from collections import Counter

pair_counts = Counter()
mismatch_rows = []

for split_name, lbl_dir in [("train", TRAIN_LBL), ("valid", VALID_LBL), ("test", TEST_LBL)]:
    split_mismatch = 0
    for fn in sorted(os.listdir(lbl_dir)):
        if not fn.endswith('.txt'):
            continue
        exp = get_expected_class(fn)
        actual = set()
        for line in open(os.path.join(lbl_dir, fn)):
            p = line.strip().split()
            if p:
                try: actual.add(int(p[0]))
                except: pass
        if exp is not None and actual and exp not in actual:
            split_mismatch += 1
            for a in actual:
                pair_counts[(CLASS_NAMES[exp], CLASS_NAMES[a])] += 1
                mismatch_rows.append({
                    'Split': split_name,
                    'File': fn,
                    'Prefix implies': CLASS_NAMES[exp],
                    'Label says': ', '.join(CLASS_NAMES[c] for c in sorted(actual))
                })
    print(f"  {split_name:>6}: {split_mismatch} mismatches")

print(f"\nTotal mismatches: {len(mismatch_rows)} / 13,143 ({100*len(mismatch_rows)/13143:.2f}%)")

print("\nMismatch breakdown by class pair:")
for (src, dst), cnt in pair_counts.most_common():
    print(f"  {cnt:4d}  {src} -> {dst}")

df_mismatch = pd.DataFrame(mismatch_rows)
print(f"\nFull mismatch table ({len(df_mismatch)} rows):")
display(df_mismatch)


## 7. EDA Summary & Model Guidance

Based on the exploration above:

1. **Dataset size (13,143 images)** — after removing 36 mislabeled files, we have **13,107 clean images**,
   placing us squarely in the 8,000–20,000 bracket. This justifies **YOLO-m** (medium) and
   **RF-DETR-base** variants per the assignment size table.

2. **All images are 640×640** — the native resolution is square, so rectangular training
   (`rect=True`) is unnecessary. Starting at 640×640 input resolution is natural.

3. **Class balance** — if imbalance exceeds 3:1 after cleaning, enabling mosaic/mixup in the
   YOLO training config will help up-sample rare classes.

4. **Object-size profile** — 88.9% of boxes are "Large" (>96² px), meaning objects fill most
   of the frame. Input resolution of 640 is sufficient; 1280 is unnecessary overhead.

5. **Boxes-per-image density** — median 1 box/image, max 49. Dense images may benefit from
   lowering NMS `iou_threshold` from 0.7 to 0.5–0.6 to reduce duplicate detections.

6. **Label quality** — 36 files (0.27%) had filename-prefix vs. label-class mismatches,
   confirmed as genuine labeling errors (not multi-object ambiguity). These are deleted in
   Section 8. The cleaned dataset is used for all model training (Notebooks 2–5).

These findings are referenced in the CONFIG cells of all four model notebooks (Notebooks 2–5).


## 8. Label Mismatch Fix — Creating a Clean Dataset

### What We Found

Every image filename contains the original **Roboflow batch/folder name** the dataset
creator used when uploading images. A systematic audit of all 13,143 label files
reveals **36 files (0.27%)** where the filename prefix implies a different class than
what the `.txt` label says:

| Filename prefix implies | `.txt` label says | Count | Verdict |
|------------------------|-------------------|-------|---------|
| Ants | **Bees** | 14 | Hard error — images visually show ants |
| Slugs | **Snails** | 6 | Ambiguous (very similar species) |
| Caterpillars | **Moths** | 6 | Ambiguous (larva vs adult) |
| Bees | **Beetles** | 4 | Hard error — images visually show bees |
| Slugs | **Ants** | 3 | Hard error — nothing alike |
| Others | Mixed | 3 | Likely errors |

| Split | Files | Mismatches |
|-------|-------|------------|
| Train | 11,502 | 27 (0.23%) |
| Valid | 1,095  | 6 (0.55%)  |
| Test  | 546    | 3 (0.55%)  |

### Fix Strategy

The cell below copies the full dataset to `/kaggle/working/AgroPest-12-Cleaned/`
and **deletes the 36 mislabeled image+label pairs** from the copy.

- The original Kaggle input path is **never modified** (it is read-only anyway)
- After this notebook runs, save the `/kaggle/working/AgroPest-12-Cleaned/` folder
  as a **new Kaggle dataset** named `AgroPest-12-Cleaned`
- All future training notebooks will add that clean dataset as input and use it
  directly — no cleaning step needed at training time

Set `APPLY_FIX = True` to execute. Safe to re-run — skips the copy if the directory
already exists.


In [ ]:
# ============================================================
# Label Fix — Copy dataset and delete mislabeled pairs
# Output: /kaggle/working/AgroPest-12-Cleaned/
# This output folder should be saved as a Kaggle dataset.
# ============================================================
import os, re, shutil

APPLY_FIX = True   # set False to dry-run without copying/deleting

DATASET_PATH = "/kaggle/input/datasets/rupankarmajumdar/crop-pests-dataset"
CLEAN_DIR    = "/kaggle/working/AgroPest-12-Cleaned"

# Filename first-word -> correct class_id
PREFIX_TO_CLASS = {
    'ant': 0, 'ants': 0, 'bee': 1, 'bees': 1, 'beetle': 2, 'beetles': 2,
    'caterpillar': 3, 'catterpillar': 3, 'earthworm': 4, 'earthworms': 4,
    'earwig': 5, 'earwigs': 5, 'grasshopper': 6, 'grasshoppers': 6,
    'moth': 7, 'moths': 7, 'slug': 8, 'slugs': 8, 'snail': 9, 'snails': 9,
    'wasp': 10, 'wasps': 10, 'weevil': 11, 'weevils': 11,
}

def _expected(fn):
    m = re.match(r'^([a-zA-Z]+)', os.path.basename(fn))
    return PREFIX_TO_CLASS.get(m.group(1).lower()) if m else None

def _actual(lbl_path):
    ids = set()
    try:
        for line in open(lbl_path):
            p = line.strip().split()
            if p:
                try: ids.add(int(p[0]))
                except: pass
    except: pass
    return ids

# ── Step 1: Identify mislabeled file stems ──
print("=" * 65)
print("STEP 1: Scanning for label mismatches")
print("=" * 65)

bad_stems = set()   # (split, stem) pairs to remove
for split in ['train', 'valid', 'test']:
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    if not os.path.exists(lbl_dir):
        print(f"  WARNING: {lbl_dir} not found")
        continue
    split_bad = 0
    for fn in sorted(os.listdir(lbl_dir)):
        if not fn.endswith('.txt'):
            continue
        exp = _expected(fn)
        act = _actual(os.path.join(lbl_dir, fn))
        if exp is not None and act and exp not in act:
            bad_stems.add((split, os.path.splitext(fn)[0]))
            split_bad += 1
    print(f"  {split:>6}: {split_bad} mismatches")

print(f"\nTotal mislabeled pairs to remove: {len(bad_stems)}")
print("Files to be removed:")
for split, stem in sorted(bad_stems):
    print(f"  [{split}] {stem[:65]}")

if not APPLY_FIX:
    print("\nDRY RUN — APPLY_FIX=False. Set to True to execute.")
else:
    # ── Step 2: Copy dataset ──
    print("\n" + "=" * 65)
    print("STEP 2: Copying dataset")
    print("=" * 65)

    if os.path.exists(CLEAN_DIR):
        print(f"Directory already exists, skipping copy: {CLEAN_DIR}")
    else:
        print(f"Copying to {CLEAN_DIR} (~537 MB, may take a minute)...")
        shutil.copytree(DATASET_PATH, CLEAN_DIR)
        print("Copy complete.")

    # ── Step 3: Delete mislabeled pairs ──
    print("\n" + "=" * 65)
    print("STEP 3: Deleting mislabeled image+label pairs")
    print("=" * 65)

    del_lbl = del_img = miss_img = 0
    for split, stem in sorted(bad_stems):
        # Delete label file
        lbl_path = os.path.join(CLEAN_DIR, split, 'labels', stem + '.txt')
        if os.path.exists(lbl_path):
            os.remove(lbl_path)
            del_lbl += 1

        # Delete image file (try common extensions)
        deleted_img = False
        for ext in ['.jpg', '.jpeg', '.png']:
            img_path = os.path.join(CLEAN_DIR, split, 'images', stem + ext)
            if os.path.exists(img_path):
                os.remove(img_path)
                del_img += 1
                deleted_img = True
                break
        if not deleted_img:
            miss_img += 1

    print(f"  Deleted {del_lbl} label files")
    print(f"  Deleted {del_img} image files")
    if miss_img:
        print(f"  WARNING: {miss_img} image files not found (labels still deleted)")

    # ── Step 4: Write data.yaml ──
    yaml_path = os.path.join(CLEAN_DIR, "data.yaml")
    with open(yaml_path, 'w') as yf:
        yf.write(
            f"train: {CLEAN_DIR}/train/images\n"
            f"val:   {CLEAN_DIR}/valid/images\n"
            f"test:  {CLEAN_DIR}/test/images\n\n"
            "nc: 12\n"
            'names: ["Ants","Bees","Beetles","Caterpillars","Earthworms","Earwigs",'
            '"Grasshoppers","Moths","Slugs","Snails","Wasps","Weevils"]\n'
        )
    print(f"\ndata.yaml written: {yaml_path}")

    # ── Step 5: Verify ──
    print("\n" + "=" * 65)
    print("STEP 5: Verification — file counts after cleaning")
    print("=" * 65)
    total_clean = 0
    for split in ['train', 'valid', 'test']:
        n_img = len([f for f in os.listdir(os.path.join(CLEAN_DIR, split, 'images'))
                     if not f.startswith('.')])
        n_lbl = len([f for f in os.listdir(os.path.join(CLEAN_DIR, split, 'labels'))
                     if f.endswith('.txt')])
        status = "OK" if n_img == n_lbl else "MISMATCH!"
        print(f"  {split:>6}: {n_img} images, {n_lbl} labels  [{status}]")
        total_clean += n_img
    print(f"\nTotal clean images: {total_clean} (removed {13143 - total_clean} from original 13,143)")

    # ── Step 6: Publish to Kaggle via API ──
    # The web UI "New Dataset" button caps at ~500 files and silently
    # truncates larger datasets. Use the Kaggle CLI instead — it zips
    # and uploads everything correctly.
    print("\n" + "=" * 65)
    print("STEP 6: Publishing AgroPest-12-Cleaned to Kaggle")
    print("=" * 65)
    import subprocess, json as _json

    # Write dataset-metadata.json (required by Kaggle CLI)
    KAGGLE_USERNAME = "sadikshahriar"   # <-- your Kaggle username
    meta = {
        "title": "AgroPest-12-Cleaned",
        "id": f"{KAGGLE_USERNAME}/agropest-12-cleaned",
        "licenses": [{"name": "MIT"}]
    }
    meta_path = os.path.join(CLEAN_DIR, "dataset-metadata.json")
    with open(meta_path, "w") as mf:
        _json.dump(meta, mf, indent=2)
    print(f"dataset-metadata.json written: {meta_path}")

    # Push via Kaggle CLI (--dir-mode zip bundles everything into one upload)
    print("\nUploading via Kaggle CLI (zips all 13,107 files)...")
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", CLEAN_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print("STDERR:", result.stderr.strip())

    if result.returncode == 0:
        print(f"\nDataset published: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/agropest-12-cleaned")
        print("Add it as input to all training notebooks (2-5).")
    else:
        print("\nUpload failed. If credentials are missing, run manually:")
        print(f"  kaggle datasets create -p {CLEAN_DIR} --dir-mode zip")
        print("Or add KAGGLE_USERNAME + KAGGLE_KEY as notebook secrets and re-run.")

    print(f"\nClean dataset location: {CLEAN_DIR}")
